## Manual analysis of total_sources 

List of sources that were selected to be part of our sample from a manual whitelist feature analysis and expansion to the whole raw sample 

### Set-up


In [1]:
import gc
import glob
import json
import os
from pathlib import Path

# Bridage des threads pour Numpy, Pandas et Scikit-Learn (laisse le CPU à DuckDB)
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import duckdb
from IPython.display import HTML, display
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

# Configuration esthétique globale
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)

print("✔ Librairies importées et environnement configuré.")

✔ Librairies importées et environnement configuré.


In [2]:
# ==============================================================================
# 1. DÉFINITION DES CHEMINS (À adapter si tes fichiers sont dans un autre dossier)
# ==============================================================================
DATA_DIR = Path("/data/gdelt/gdelt_parquet_db")
SOURCE_MAP_PATH = Path("/data/gdelt/gdelt_sources_mapping.json")
RETAINED_IDS_PATH = Path("liste_ids_retenus.txt")
DOMAINS_PARQUET_PATH = "data/domains/domains_*.parquet"

# ==============================================================================
# 2. INITIALISATION ET PARAMÉTRAGE DE DUCKDB
# ==============================================================================
con = duckdb.connect()

# Réglage musclé pour ton grand serveur partagé (128 Go RAM / 16 threads en standard)
con.execute("PRAGMA memory_limit='128GB'")
con.execute("PRAGMA threads=16")

# 3. Chargement du dictionnaire JSON (Traduction ID <-> Nom de domaine)
with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
  source_map = json.load(f)

src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in source_map["id_to_source"].keys()],
    "SourceCommonName": list(source_map["id_to_source"].values()),
})
con.register("src_map", src_df)

# 4. Chargement de TA LISTE PROPRE d'ID retenus dans une table DuckDB dédiée
con.execute(f"""
    CREATE OR REPLACE TABLE retained_ids AS 
    SELECT column0::BIGINT AS id 
    FROM read_csv('{RETAINED_IDS_PATH}', header=False)
""")

nb_ids = con.execute("SELECT COUNT(*) FROM retained_ids").fetchone()[0]
print(
    f"✔ DuckDB initialisé. Table 'retained_ids' chargée ({nb_ids:,} médias"
    " légitimes)."
)

✔ DuckDB initialisé. Table 'retained_ids' chargée (13,334 médias légitimes).


In [3]:
glob_pattern = str(DATA_DIR / "gdelt_*.parquet")

print(
    "⏳ Création de la vue maîtresse 'gkg_clean' (Filtrage sur échantillon"
    " propre & enrichissement)..."
)

con.execute(f"""
    CREATE OR REPLACE VIEW gkg_clean AS
    
    WITH raw_filtered AS (
        -- 1. Nettoyage initial : dates valides, exclusion ligne corrompue et thèmes vides
        SELECT *
        FROM read_parquet('{glob_pattern}')
        WHERE regexp_matches(CAST(DATE AS VARCHAR), '^\d{{14}}$')
          AND GKGRECORDID != '20210925181500-T1111'
          AND EnhancedThemes IS NOT NULL 
          AND EnhancedThemes != ''
    ),
    
    repaired AS (
        -- 2. Réparation de l'identifiant (si 0 ou NULL, on tente de matcher le nom de domaine via src_map)
        SELECT 
            r.* EXCLUDE (SourceCommonName_ID),
            CASE 
                WHEN COALESCE(r.SourceCommonName_ID, 0) = 0 THEN m.SourceCommonName_ID
                ELSE r.SourceCommonName_ID
            END AS SourceCommonName_ID
        FROM raw_filtered r
        LEFT JOIN src_map m 
          ON RTRIM(regexp_extract(r.DocumentIdentifier, 'https?://(?:www\.)?([^/?:]+)', 1), '.') = m.SourceCommonName
    )
    
    -- 3. FILTRAGE STRICT SUR L'ÉCHANTILLON ET ENRICHISSEMENT WIKIDATA
    SELECT 
        rep.*,
        w.medialabel,
        w.typelabel,
        w.countrylabel,
        w.inception,
        CASE WHEN w.Src_ID IS NOT NULL THEN 1 ELSE 0 END AS is_wiki
    FROM repaired rep
    -- 🔥 LE FILTRE CLÉ : Jointure interne avec ta liste des sources retenues !
    INNER JOIN retained_ids rid 
      ON rep.SourceCommonName_ID = rid.id
    -- Enrichissement optionnel : si la source est dans Wikidata, on récupère ses labels
    LEFT JOIN (
        SELECT 
            id AS Src_ID, 
            medialabel, 
            typelabel, 
            countrylabel, 
            inception
        FROM read_parquet('{DOMAINS_PARQUET_PATH}')
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY inception ASC, countrylabel ASC) = 1
    ) w ON rep.SourceCommonName_ID = w.Src_ID;
""")

print(
    "✔ Vue 'gkg_clean' prête ! Elle ne contient que les articles de ton"
    " échantillon sélectionné."
)

⏳ Création de la vue maîtresse 'gkg_clean' (Filtrage sur échantillon propre & enrichissement)...
✔ Vue 'gkg_clean' prête ! Elle ne contient que les articles de ton échantillon sélectionné.


<>:57: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\.'
<>:57: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_3419785/1892980978.py:57: SyntaxWarning: invalid escape sequence '\d'
  """)
/tmp/ipykernel_3419785/1892980978.py:57: SyntaxWarning: invalid escape sequence '\.'
  """)


In [ ]:
print("⏳ Calcul de la volumétrie globale sur ton échantillon...")

# 1. Requête rapide de vérification
df_check = con.execute("""
    SELECT 
        COUNT(*) AS total_articles_disponibles,
        COUNT(DISTINCT SourceCommonName_ID) AS medias_actifs,
        MIN(strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE) AS date_min,
        MAX(strptime(substr(CAST(DATE AS VARCHAR), 1, 8), '%Y%m%d')::DATE) AS date_max,
        SUM(is_wiki) AS articles_avec_metadata_wiki
    FROM gkg_clean;
""").df()

# 2. Affichage du résumé
display(df_check.style.format({
    "total_articles_disponibles": "{:,.0f}",
    "medias_actifs": "{:,.0f}",
    "articles_avec_metadata_wiki": "{:,.0f}"
}))

# 3. Aperçu des 3 premiers articles au hasard pour vérifier la structure
print("\n👀 Aperçu de 3 articles au hasard dans la vue propre :")
display(con.execute("""
    SELECT 
        GKGRECORDID, 
        SourceCommonName_ID, 
        medialabel, 
        countrylabel, 
        substr(DocumentIdentifier, 1, 60) || '...' AS URL,
        substr(EnhancedThemes, 1, 80) || '...' AS Themes_extrait
    FROM gkg_clean 
    USING SAMPLE 3;
""").df())

⏳ Calcul de la volumétrie globale sur ton échantillon...


### Random sample analysis 

#### Analysis of 100 sources, randomly chosen among all the selected sources 

In [7]:
print("Tirage aléatoire de 1000 sources dans l'échantillon retenu...\n")

# 1. Requête SQL : Jointure avec le dictionnaire des noms et tirage aléatoire
df_sample_100 = con.execute("""
    SELECT 
        r.id AS SourceCommonName_ID,
        COALESCE(m.SourceCommonName, 'Domaine inconnu (' || r.id || ')') AS SourceCommonName
    FROM retained_ids r
    LEFT JOIN src_map m ON r.id = m.SourceCommonName_ID
    ORDER BY random()
    LIMIT 1000;
""").df()

# 2. Réinitialisation de l'index pour avoir un comptage propre de 1 à 100
df_sample_100.index = range(1, len(df_sample_100) + 1)
df_sample_100.index.name = '#'

# 3. AFFICHAGE EN GRILLE DE TEXTE (4 colonnes) pour un balayage visuel ultra-rapide
domaines = df_sample_100['SourceCommonName'].tolist()
n_cols = 4
n_rows = (len(domaines) + n_cols - 1) // n_cols

print("BALAYAGE RAPIDE — 1000 médias tirés au hasard :\n" + "="*85)
for r in range(n_rows):
    row_items = []
    for c in range(n_cols):
        idx = r + c * n_rows
        if idx < len(domaines):
            # On formate chaque nom sur 20 caractères pour aligner les colonnes
            row_items.append(f"{idx+1:3d}. {domaines[idx]:<18}")
    print(" | ".join(row_items))

print("\n" + "="*85)

# 4. AFFICHAGE DU DATAFRAME COMPLET (Pour inspection détaillée si besoin)
# On force Pandas à afficher les 100 lignes sans tronquer
with pd.option_context('display.max_rows', 1000):
    display(df_sample_100.style.set_properties(**{
        'font-weight': 'bold', 
        'text-align': 'left'
    }))

Tirage aléatoire de 1000 sources dans l'échantillon retenu...

BALAYAGE RAPIDE — 1000 médias tirés au hasard :
  1. filkhbr.com        | 251. toyokeizai.net     | 501. government.ru      | 751. eldiariocba.com.ar
  2. wienerzeitung.at   | 252. customstoday.com.pk | 502. myheraldreview.com | 752. dumbartonreporter.co.uk
  3. mininggazette.com  | 253. aldo2.com          | 503. sluggerotoole.com  | 753. 3djuegos.com      
  4. codiceinformativo.com | 254. apa.az             | 504. pudahuel.cl        | 754. kenilworthweeklynews.co.uk
  5. bonde.com.br       | 255. wixx.com           | 505. droitwichstandard.co.uk | 755. ccenterdispatch.com
  6. contrepoints.org   | 256. charter97.org      | 506. kincardineshireobserver.co.uk | 756. latribune.fr      
  7. thevwindependent.com | 257. trinidadexpress.com | 507. boundarysentinel.com | 757. nczas.info        
  8. ziarulfaclia.ro    | 258. bicesteradvertiser.net | 508. gwinnettdailypost.com | 758. dqdaily.com       
  9. njtoday.net        | 2

,SourceCommonName_ID,SourceCommonName
#,,
1,32454,filkhbr.com
2,7348,wienerzeitung.at
3,4303,mininggazette.com
4,28628,codiceinformativo.com
5,25406,bonde.com.br
6,31506,contrepoints.org
7,7625,thevwindependent.com
8,28888,ziarulfaclia.ro
9,23351,njtoday.net


"You are a research assistant in economics and I give you this list of 1000 sources randmoly chosen from my sample. I want to know how many are not newspapers that discuss economic, politic and business topics"


In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import ssl
import urllib.error
import urllib.request
from bs4 import BeautifulSoup
import pandas as pd

print(
    "⏳ [1/4] Extraction des domaines retenus depuis DuckDB pour le scraping..."
)

# 1. On récupère la liste exacte de tes ID retenus et leurs noms de domaine
df_to_scrape = con.execute("""
    SELECT 
        r.id AS SourceCommonName_ID,
        m.SourceCommonName AS domain
    FROM retained_ids r
    INNER JOIN src_map m ON r.id = m.SourceCommonName_ID
    WHERE m.SourceCommonName IS NOT NULL
      AND m.SourceCommonName != ''
""").df()

print(f"✔ {len(df_to_scrape):,} domaines uniques à analyser.")

# ==============================================================================
# 2. DICTIONNAIRE SÉMANTIQUE MULTILINGUE DE L'EXCLUSION (EN, FR, ES, DE)
# ==============================================================================
# Attention à ne pas mettre de mots trop génériques (ex: "market" qui bloquerait la presse financière !)
NON_NEWS_KEYWORDS = {
    "Gaming / Jeux": [
        "game",
        "gaming",
        "juegos",
        "videojuegos",
        "gamer",
        "nintendo",
        "playstation",
        "xbox",
        "esports",
        "mmorpg",
        "jeux vidéo",
    ],
    "Lifestyle / Mode / People": [
        "fashion",
        "beauty",
        "lifestyle",
        "moda",
        "belleza",
        "maquillaje",
        "horoscope",
        "astrology",
        "astrologie",
        "gossip",
        "celebrity",
        "people",
        "peoplenews",
        "recetas",
        "recipe",
        "cooking",
        "cuisine",
        "gourmet",
    ],
    "Académique / Scientifique pur": [
        "journal of",
        "proceedings of",
        "chemistry",
        "physics",
        "biology",
        "clinical",
        "surgery",
        "molecular",
        "ieee",
        "acs.",
        "scientific journal",
        "peer-reviewed",
    ],
    "Shopping / Commercial / Forum": [
        "coupon",
        "promo code",
        "shop online",
        "tienda online",
        "boutique en ligne",
        "e-commerce",
        "forum de discussion",
        "message board",
    ],
}

# Configuration SSL permissive (beaucoup de sites de presse ont des certificats mal configurés)
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE


# ==============================================================================
# 3. FONCTION DE SCRAPING UNITAIRE (ROBUSTE AUX ERREURS)
# ==============================================================================
def inspect_domain(row):
  domain = row["domain"]
  src_id = row["SourceCommonName_ID"]
  url = f"https://{domain}"

  # Headers modernes pour simuler un vrai navigateur et éviter les blocages Cloudflare
  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
          " like Gecko) Chrome/120.0.0.0 Safari/537.36"
      ),
      "Accept": (
          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
      ),
      "Accept-Language": "en-US,en;q=0.5,fr;q=0.3",
  }

  try:
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=4, context=ssl_ctx) as response:
      # On ne lit que les premiers 100 Ko pour aller très vite (les métadonnées sont dans le <head>)
      html = response.read(100000).decode("utf-8", errors="ignore")

    soup = BeautifulSoup(html, "html.parser")

    # Extraction combinée : <title> + <meta name="description"> + <meta property="og:description">
    title = soup.title.string if soup.title else ""
    meta_desc = soup.find("meta", attrs={"name": re.compile(r"^description$", re.I)})
    og_desc = soup.find("meta", attrs={"property": re.compile(r"^og:description$", re.I)})

    text_to_check = (
        f"{title} {meta_desc['content'] if meta_desc and 'content' in meta_desc.attrs else ''} {og_desc['content'] if og_desc and 'content' in og_desc.attrs else ''}"
    ).lower()

    # Recherche des mots-clés interdits
    for categorie, mots in NON_NEWS_KEYWORDS.items():
      for mot in mots:
        # Recherche en mot entier pour éviter d'exclure un mot contenant la chaîne
        if re.search(r"\b" + re.escape(mot) + r"\b", text_to_check):
          return {
              "SourceCommonName_ID": src_id,
              "domain": domain,
              "status": "❌ Exclus",
              "motif": f"{categorie} (mot: '{mot}')",
              "extrait_meta": text_to_check[:120].strip(),
          }

    return {
        "SourceCommonName_ID": src_id,
        "domain": domain,
        "status": "✔ Presse Légitime",
        "motif": "OK",
        "extrait_meta": text_to_check[:120].strip(),
    }

  except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError):
    # En cas d'échec HTTPS, on tente rapidement en HTTP simple avant d'abandonner
    try:
      req = urllib.request.Request(f"http://{domain}", headers=headers)
      with urllib.request.urlopen(
          req, timeout=3, context=ssl_ctx
      ) as response:
        html = response.read(100000).decode("utf-8", errors="ignore")
      soup = BeautifulSoup(html, "html.parser")
      title = soup.title.string if soup.title else ""
      return {
          "SourceCommonName_ID": src_id,
          "domain": domain,
          "status": "✔ Presse Légitime",
          "motif": "OK (via HTTP)",
          "extrait_meta": (str(title)[:120] if title else ""),
      }
    except Exception:
      pass

    return {
        "SourceCommonName_ID": src_id,
        "domain": domain,
        "status": "⚠ Inaccessible / Bloqué",
        "motif": "Timeout / Cloudflare / Paywall",
        "extrait_meta": "",
    }
  except Exception as e:
    return {
        "SourceCommonName_ID": src_id,
        "domain": domain,
        "status": "⚠ Inaccessible / Bloqué",
        "motif": "Erreur technique",
        "extrait_meta": "",
    }


# ==============================================================================
# 4. EXÉCUTION EN PARALLÈLE (25 THREADS SIMULTANÉS)
# ==============================================================================
print(
    "⏳ [2/4] Lancement du crawler sur 25 threads (compter ~2 minutes)..."
)

results = []
rows_dict = df_to_scrape.to_dict(orient="records")

with ThreadPoolExecutor(max_workers=25) as executor:
  futures = {executor.submit(inspect_domain, row): row for row in rows_dict}
  completed = 0
  for future in as_completed(futures):
    results.append(future.result())
    completed += 1
    if completed % 250 == 0 or completed == len(rows_dict):
      print(
          f"   ➔ Progression : {completed:,} / {len(rows_dict):,} domaines"
          " analysés..."
      )

df_results = pd.DataFrame(results)

print("\n📊 [3/4] RÉSULTAT DE LA CLASSIFICATION ÉDITORIALE :")
display(
    df_results["status"].value_counts().rename("Nombre de domaines").to_frame()
)

# ==============================================================================
# 5. DIAGNOSTIC DES EXCLUSIONS POUR VALIDER qualitativement
# ==============================================================================
df_exclus = df_results[df_results["status"] == "❌ Exclus"].copy()
print(
    f"\n🚫 [4/4] LISTE DES SITES DÉTECTÉS COMME HORS-SUJET ({len(df_exclus):,}"
    " domaines) :"
)
print("=" * 95)

if len(df_exclus) > 0:
  with pd.option_context("display.max_rows", 100):
    display(
        df_exclus[["domain", "motif", "extrait_meta"]]
        .sort_values(by="motif")
        .style.set_properties(**{"text-align": "left", "font-size": "11px"})
    )
else:
  print("Aucun site exclu avec les mots-clés actuels !")

# Sauvegarde de ce diagnostic en CSV pour audit
df_results.to_csv("audit_classification_domaines.csv", index=False)
print(
    "\n✔ Fichier 'audit_classification_domaines.csv' sauvegardé pour analyse."
)

⏳ [1/4] Extraction des domaines retenus depuis DuckDB pour le scraping...
✔ 13,667 domaines uniques à analyser.
⏳ [2/4] Lancement du crawler sur 25 threads (compter ~2 minutes)...
   ➔ Progression : 250 / 13,667 domaines analysés...
   ➔ Progression : 500 / 13,667 domaines analysés...
   ➔ Progression : 750 / 13,667 domaines analysés...
   ➔ Progression : 1,000 / 13,667 domaines analysés...
   ➔ Progression : 1,250 / 13,667 domaines analysés...
   ➔ Progression : 1,500 / 13,667 domaines analysés...
   ➔ Progression : 1,750 / 13,667 domaines analysés...
   ➔ Progression : 2,000 / 13,667 domaines analysés...
   ➔ Progression : 2,250 / 13,667 domaines analysés...
   ➔ Progression : 2,500 / 13,667 domaines analysés...
   ➔ Progression : 2,750 / 13,667 domaines analysés...
   ➔ Progression : 3,000 / 13,667 domaines analysés...
   ➔ Progression : 3,250 / 13,667 domaines analysés...
   ➔ Progression : 3,500 / 13,667 domaines analysés...
   ➔ Progression : 3,750 / 13,667 domaines analysés...
 

,Nombre de domaines
status,
✔ Presse Légitime,9257
⚠ Inaccessible / Bloqué,3921
❌ Exclus,489



🚫 [4/4] LISTE DES SITES DÉTECTÉS COMME HORS-SUJET (489 domaines) :


,domain,motif,extrait_meta
196,digitaljournal.com,Académique / Scientifique pur (mot: 'journal of'),digital journal | the journal of record for technology decisions in canada we cover innovation for the leaders who own t
2033,sanjuanjournal.com,Académique / Scientifique pur (mot: 'journal of'),your local homepage. | the journal of the san juan islands your local homepage. your local homepage.
2211,onlineopinion.com.au,Académique / Scientifique pur (mot: 'journal of'),on line opinion - australia's e-journal of social and political debate on line opinion - australia's e-journal of social
4047,theartnewspaper.com,Académique / Scientifique pur (mot: 'journal of'),the art newspaper - international art news and events the art newspaper is the journal of record for the visual arts wor
5631,asiaasset.com,Académique / Scientifique pur (mot: 'journal of'),home - asia asset management news asia asset management – the journal of investments and pensions – was established in 1
13070,oakpark.com,Académique / Scientifique pur (mot: 'journal of'),"wednesday journal of oak park and river forest | cook county, illinois weekly newspaper from growing community media ser"
13312,respect-mag.com,Académique / Scientifique pur (mot: 'journal of'),respect. | the photo journal of hip-hop culture - the photo journal of hip-hop culture the photo journal of hip-hop cult
4659,physicsworld.com,Académique / Scientifique pur (mot: 'physics'),home – physics world physics world represents a key part of iop publishing's mission to communicate world-class research
2355,segre.com,Gaming / Jeux (mot: 'esports'),"segre - digital líder a les comarques de lleida notícies d'última hora, actualitat, esports, política, imatges i vídeos"
6532,vilaweb.cat,Gaming / Jeux (mot: 'esports'),"vilaweb - diari digital líder en català. última hora, notícies, opinió i vídeos notícies nacionals andorra, catalunya, p"



✔ Fichier 'audit_classification_domaines.csv' sauvegardé pour analyse.


In [13]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import ssl
import urllib.error
import urllib.request
from bs4 import BeautifulSoup
import pandas as pd

print(
    "⏳ [1/4] Extraction des domaines pour arbitrage (Guillotine Anti-Niches)..."
)

df_to_scrape = con.execute("""
    SELECT 
        r.id AS SourceCommonName_ID,
        m.SourceCommonName AS domain
    FROM retained_ids r
    INNER JOIN src_map m ON r.id = m.SourceCommonName_ID
    WHERE m.SourceCommonName IS NOT NULL AND m.SourceCommonName != ''
""").df()

print(f"✔ {len(df_to_scrape):,} domaines à passer au scanner.")

# ==============================================================================
# 2. LA GUILLOTINE STRICTE (ZÉRO TOLÉRANCE — TUE MÊME S'IL Y A LE MOT "NEWS")
# ==============================================================================
STRICT_KILL_KEYWORDS = {
    "Gaming & Esports": [
        "gaming",
        "gamer",
        "video game",
        "video games",
        "jeux vidéo",
        "jeux video",
        "videojuego",
        "videojuegos",
        "jogos",
        "nintendo",
        "playstation",
        "xbox",
        "esports",
        "mmorpg",
        "rpg",
        "gameplay",
        "console",
        "ign.",
        "kotaku",
        "3djuegos",
        "board game",
        "tabletop",
        "pc gaming",
        "steam",
    ],
    "Tech Gadgets & Hardware Reviews": [
        "gadget reviews",
        "hardware reviews",
        "smartphone reviews",
        "tech reviews",
        "pc builds",
        "overclocking",
        "gpu",
        "cpu",
        "apple watch review",
        "phone review",
    ],
    "Fashion / Gossip / Celebrities": [
        "fashion magazine",
        "haute couture",
        "celebrity gossip",
        "gossip",
        "celebrity news",
        "people news",
        "paparazzi",
        "beauty tips",
        "makeup",
        "belleza y moda",
        "mode et beauté",
        "k-pop",
    ],
    "Cuisine / Recettes / Food": [
        "recipes",
        "recetas",
        "recettes",
        "cooking",
        "cuisine",
        "baking",
        "food blog",
        "wine tasting",
        "gourmet recipes",
        "foodie",
    ],
    "Crypto / Betting / Casino": [
        "nft",
        "crypto trading",
        "tokenomics",
        "online casino",
        "sports betting",
        "poker online",
        "apuestas",
        "cryptocurrency news",
        "bitcoin news",
    ],
    "Académique Pur": [
        "peer-reviewed",
        "scientific journal",
        "proceedings of",
        "clinical trials",
        "academic journal",
        "journal of chemistry",
        "journal of physics",
        "journal of biology",
        "journal of medicine",
    ],
    "Forums & Coupons": [
        "coupon code",
        "promo code",
        "discount codes",
        "vbulletin",
        "discussion forum",
        "message board",
        "subreddit",
    ],
}

# ==============================================================================
# 3. LE BOUCLIER (Ne sauve QUE les mots ambigus comme "culture" ou "style")
# ==============================================================================
POSITIVE_NEWS_KEYWORDS = [
    "breaking news",
    "latest news",
    "daily news",
    "local news",
    "world news",
    "national news",
    "journalism",
    "newspaper",
    "politics",
    "economy",
    "business",
    "finance",
    "financial",
    "investing",
    "stock market",
    "actualité",
    "actualités",
    "information",
    "quotidien",
    "société",
    "politique",
    "économie",
    "actualidad",
    "noticias",
    "diario",
    "periodico",
    "economia",
    "finanzas",
    "nachrichten",
    "zeitung",
    "wirtschaft",
    "politik",
    "giornale",
    "notizie",
]

# Mots généralistes ambigus (qui peuvent être dans un journal OU dans un magazine spécialisé)
SOFT_KILL_KEYWORDS = [
    "style",
    "entertainment",
    "culture",
    "sports",
    "automobile",
    "voyage",
    "travel",
    "magazine",
    "decor",
]

ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE


# ==============================================================================
# 4. FONCTION D'ARBITRAGE SANS FAILLE
# ==============================================================================
def inspect_domain(row):
  domain = row["domain"]
  src_id = row["SourceCommonName_ID"]
  url = f"https://{domain}"

  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
          " like Gecko) Chrome/120.0.0.0 Safari/537.36"
      ),
      "Accept-Language": "en-US,en;q=0.9,fr;q=0.8,es;q=0.7,de;q=0.6",
  }

  try:
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=4, context=ssl_ctx) as response:
      html = response.read(100000).decode("utf-8", errors="ignore")

    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.string if soup.title else ""
    meta_desc = soup.find(
        "meta", attrs={"name": re.compile(r"^description$", re.I)}
    )
    og_desc = soup.find(
        "meta", attrs={"property": re.compile(r"^og:description$", re.I)}
    )

    text_to_check = (
        f"{title} {meta_desc['content'] if meta_desc and 'content' in meta_desc.attrs else ''} {og_desc['content'] if og_desc and 'content' in og_desc.attrs else ''}"
    ).lower()

    # --- ÉTAPE 1 : LA GUILLOTINE STRICTE (TUE IMMÉDIATEMENT — PAS DE PARDON POSITIF !) ---
    for cat, mots in STRICT_KILL_KEYWORDS.items():
      for mot in mots:
        if re.search(r"\b" + re.escape(mot) + r"\b", text_to_check):
          return {
              "SourceCommonName_ID": src_id,
              "domain": domain,
              "status": "❌ Exclus",
              "motif": f"Guillotine {cat} ('{mot}')",
              "extrait_meta": text_to_check[:110].strip(),
          }

    # --- ÉTAPE 2 : RECHERCHE D'UN VRAI CONTEXTE DE PRESSE GÉNÉRALISTE ---
    has_news_keywords = any(
        re.search(r"\b" + re.escape(pos) + r"\b", text_to_check)
        for pos in POSITIVE_NEWS_KEYWORDS
    )

    # --- ÉTAPE 3 : ARBITRAGE DES MOTS DOUX (Culture, Style, Entertainment...) ---
    for mot_doux in SOFT_KILL_KEYWORDS:
      if re.search(r"\b" + re.escape(mot_doux) + r"\b", text_to_check):
        if has_news_keywords:
          return {
              "SourceCommonName_ID": src_id,
              "domain": domain,
              "status": "✔ Presse Légitime",
              "motif": (
                  f"Sauvé par Contexte News (malgré mot généraliste"
                  f" '{mot_doux}')"
              ),
              "extrait_meta": text_to_check[:110].strip(),
          }
        else:
          return {
              "SourceCommonName_ID": src_id,
              "domain": domain,
              "status": "❌ Exclus",
              "motif": (
                  f"Magazine de loisirs sans contexte d'actu ('{mot_doux}')"
              ),
              "extrait_meta": text_to_check[:110].strip(),
          }

    return {
        "SourceCommonName_ID": src_id,
        "domain": domain,
        "status": "✔ Presse Légitime",
        "motif": "OK (Standard)",
        "extrait_meta": text_to_check[:110].strip(),
    }

  except Exception:
    return {
        "SourceCommonName_ID": src_id,
        "domain": domain,
        "status": "✔ Presse Légitime",
        "motif": "OK (Inaccessible web)",
        "extrait_meta": "",
    }


# ==============================================================================
# 5. EXÉCUTION EN PARALLÈLE ET DIAGNOSTIC DES REJETS
# ==============================================================================
print("⏳ [2/4] Lancement de la Guillotine Anti-Niches sur 25 threads...")
results = []
rows_dict = df_to_scrape.to_dict(orient="records")

with ThreadPoolExecutor(max_workers=25) as executor:
  futures = {executor.submit(inspect_domain, row): row for row in rows_dict}
  for i, future in enumerate(as_completed(futures), 1):
    results.append(future.result())
    if i % 250 == 0 or i == len(rows_dict):
      print(f"   ➔ Progression : {i:,} / {len(rows_dict):,} domaines...")

df_results = pd.DataFrame(results)

print("\n📊 [3/4] BILAN DE LA GUILLOTINE ANTI-NICHES :")
display(
    df_results["status"].value_counts().rename("Nombre de domaines").to_frame()
)

# 6. Affichage par catégorie d'exclusion
df_exclus = df_results[df_results["status"] == "❌ Exclus"].copy()
print(
    f"\n🚫 [4/4] RÉPARTITION DES NICHES ÉVINCÉES ({len(df_exclus):,} sites"
    " nettoyés) :"
)
print("=" * 95)
if len(df_exclus) > 0:
  # On extrait la catégorie du motif (ex: "Guillotine Gaming & Esports ('gaming')")
  df_exclus["Categorie_Rejet"] = (
      df_exclus["motif"]
      .str.extract(r"Guillotine ([^(]+)", expand=False)
      .fillna("Magazine Loisirs")
  )
  display(
      df_exclus["Categorie_Rejet"]
      .value_counts()
      .rename("Nombre de sites exclus")
      .to_frame()
      .style.background_gradient(cmap="OrRd")
  )

  print("\n👀 EXEMPLES DE SITES DE GAMING / TECH / FOOD ÉLIMINÉS :")
  display(
      df_exclus[["domain", "motif", "extrait_meta"]]
      .sort_values(by="motif")
      .head(30)
      .style.set_properties(**{"text-align": "left"})
  )

df_results.to_csv("audit_classification_domaines_v4_guillotine.csv", index=False)
print(
    "✔ Fichier 'audit_classification_domaines_v4_guillotine.csv' sauvegardé"
    " avec succès !"
)

⏳ [1/4] Extraction des domaines pour arbitrage (Guillotine Anti-Niches)...
✔ 13,667 domaines à passer au scanner.
⏳ [2/4] Lancement de la Guillotine Anti-Niches sur 25 threads...
   ➔ Progression : 250 / 13,667 domaines...
   ➔ Progression : 500 / 13,667 domaines...
   ➔ Progression : 750 / 13,667 domaines...
   ➔ Progression : 1,000 / 13,667 domaines...
   ➔ Progression : 1,250 / 13,667 domaines...
   ➔ Progression : 1,500 / 13,667 domaines...
   ➔ Progression : 1,750 / 13,667 domaines...
   ➔ Progression : 2,000 / 13,667 domaines...
   ➔ Progression : 2,250 / 13,667 domaines...
   ➔ Progression : 2,500 / 13,667 domaines...
   ➔ Progression : 2,750 / 13,667 domaines...
   ➔ Progression : 3,000 / 13,667 domaines...
   ➔ Progression : 3,250 / 13,667 domaines...
   ➔ Progression : 3,500 / 13,667 domaines...
   ➔ Progression : 3,750 / 13,667 domaines...
   ➔ Progression : 4,000 / 13,667 domaines...
   ➔ Progression : 4,250 / 13,667 domaines...
   ➔ Progression : 4,500 / 13,667 domaines...

/tmp/ipykernel_2371148/2265349876.py:204: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "html.parser")


   ➔ Progression : 13,500 / 13,667 domaines...
   ➔ Progression : 13,667 / 13,667 domaines...

📊 [3/4] BILAN DE LA GUILLOTINE ANTI-NICHES :


,Nombre de domaines
status,
✔ Presse Légitime,13231
❌ Exclus,436



🚫 [4/4] RÉPARTITION DES NICHES ÉVINCÉES (436 sites nettoyés) :


,Nombre de sites exclus
Categorie_Rejet,
Magazine Loisirs,338
Gaming & Esports,37
Fashion / Gossip / Celebrities,36
Cuisine / Recettes / Food,16
Crypto / Betting / Casino,5
Tech Gadgets & Hardware Reviews,3
Forums & Coupons,1



👀 EXEMPLES DE SITES DE GAMING / TECH / FOOD ÉLIMINÉS :


,domain,motif,extrait_meta
12796,kaperadio1550.com,Guillotine Crypto / Betting / Casino ('apuestas'),apostala paraguay online | bonos exclusivos y juegos en línea disfruta de la mejor experiencia en apostala par
13256,thebitcoinnews.com,Guillotine Crypto / Betting / Casino ('bitcoin news'),thebitcoinnews.com - the bitcoin news
12027,banklesstimes.com,Guillotine Crypto / Betting / Casino ('cryptocurrency news'),"cryptocurrency news, reviews & cryptocurrency buying guides | banklesstimes"
8742,gazeta.lviv.ua,Guillotine Crypto / Betting / Casino ('online casino'),онлайн казино україни 2026 ξ рейтинг ліцензійних інтернет-казино на гроші найкращі ліцензійні 【 онлайн казино
5637,nigerianpilot.com,Guillotine Crypto / Betting / Casino ('sports betting'),best sports betting site in nigeria + top 5 - nigerian pilot ranking of the best sports betting sites in niger
3363,taiwannews.com.tw,Guillotine Cuisine / Recettes / Food ('cuisine'),"taiwan news - voice of the people, bridge to the world taiwan news provides the latest stories on all things r"
6840,bolivia.com,Guillotine Cuisine / Recettes / Food ('recetas'),bolivia.com | últimas noticias y servicios de bolivia el portal que une a los bolivianos. información general
13302,directoalpaladar.com,Guillotine Cuisine / Recettes / Food ('recetas'),"recetas de cocina, postres y gastronomía. directo al paladar (dap) recetas de cocina y gastronomía. las mejore"
8088,madame.lefigaro.fr,Guillotine Cuisine / Recettes / Food ('recettes'),"mode, beauté, recettes, société, horoscope, célébrités"
8804,elle.fr,Guillotine Cuisine / Recettes / Food ('recettes'),"magazine elle : magazine feminin mode, beauté, cuisine - elle magazine de mode créé par les femmes pour les"


✔ Fichier 'audit_classification_domaines_v4_guillotine.csv' sauvegardé avec succès !


In [11]:
import pandas as pd
from IPython.display import display

print("⏳ Chargement des sources exclues par l'arbitrage textuel pur...")

# 1. Chargement depuis la mémoire ou depuis votre fichier de sauvegarde
if "df_results" in globals():
  df_audit = df_results.copy()
else:
  df_audit = pd.read_csv("audit_classification_domaines_pure_text.csv")

# 2. Isolation stricte des médias classés "❌ Exclus"
df_exclus = df_audit[df_audit["status"] == "❌ Exclus"].copy()

print(f"✔ Nombre total de médias exclus : {len(df_exclus):,}\n" + "=" * 85)

# ==============================================================================
# 3. RÉPARTITION MACRO : POURQUOI ONT-ILS ÉTÉ RETIRÉS ?
# ==============================================================================
print("📊 RÉPARTITION PAR MOTIF D'EXCLUSION :")
display(
    df_exclus["motif"]
    .value_counts()
    .rename("Nombre de sites")
    .to_frame()
    .style.background_gradient(cmap="OrRd")
)

# ==============================================================================
# 4. AUDIT VISUEL DÉTAILLÉ (Domaine + Motif + Extrait HTML)
# ==============================================================================
print(
    "\n🚫 LISTE DÉTAILLÉE DES MÉDIAS RETIRÉS (Triée par motif puis par nom de"
    " domaine) :"
)
print("=" * 85)

if len(df_exclus) > 0:
  # On autorise l'affichage jusqu'à 300 lignes pour balayer tout votre échantillon d'un coup d'œil
  with pd.option_context("display.max_rows", 300, "display.max_colwidth", 110):
    display(
        df_exclus[["domain", "motif", "extrait_meta"]]
        .sort_values(by=["motif", "domain"])
        .style.set_properties(
            **{
                "text-align": "left",
                "font-size": "11px",
                "border-bottom": "1px solid #eee",
            }
        )
        .set_properties(subset=["domain"], **{"font-weight": "bold"})
    )
else:
  print("Aucun média n'a été retiré avec cette configuration !")

⏳ Chargement des sources exclues par l'arbitrage textuel pur...
✔ Nombre total de médias exclus : 143
📊 RÉPARTITION PAR MOTIF D'EXCLUSION :


,Nombre de sites
motif,
Spécialisé Lifestyle/Fashion sans contexte news ('lifestyle'),94
Spécialisé Lifestyle/Fashion sans contexte news ('fashion'),14
Rejet Absolu : Astrologie / Horoscope ('astrology'),6
Rejet Absolu : Gaming / Jeux ('videojuegos'),6
Spécialisé Lifestyle/Fashion sans contexte news ('beauty'),5
Rejet Absolu : Gaming / Jeux ('esports'),4
Rejet Absolu : Astrologie / Horoscope ('horoscope'),4
Spécialisé Lifestyle/Fashion sans contexte news ('celebrity news'),2
Rejet Absolu : Gaming / Jeux ('xbox'),2



🚫 LISTE DÉTAILLÉE DES MÉDIAS RETIRÉS (Triée par motif puis par nom de domaine) :


,domain,motif,extrait_meta
13627,andhrajyothy.com,Rejet Absolu : Astrologie / Horoscope ('astrology'),"andhrajyothi telugu news: latest telugu news , latest తెలుగు వార్తలు and live updates | breaking news in ap an"
5521,greatandhra.com,Rejet Absolu : Astrologie / Horoscope ('astrology'),"greatandhra | telugu news, politics and movies - greatandhra greatandhra - world no 1 leading telugu daily new"
978,manoramaonline.com,Rejet Absolu : Astrologie / Horoscope ('astrology'),manorama online: malayalam breaking news | kerala local news | politics | videos | podcasts | business | spor
3728,prokerala.com,Rejet Absolu : Astrologie / Horoscope ('astrology'),"prokerala – health, ayurveda, travel, astrology, ringtones, news"
7403,vaartha.com,Rejet Absolu : Astrologie / Horoscope ('astrology'),latest telugu news | breaking news telugu | telugu news-vaartha vaartha (వార్త) covers today's latest telugu n
7333,vikatan.com,Rejet Absolu : Astrologie / Horoscope ('astrology'),தமிழ் செய்திகள் | tamil news online | latest breaking news in tamil | live updates – vikatan get the latest ta
2160,echoofindia.com,Rejet Absolu : Astrologie / Horoscope ('horoscope'),"the echo of india | read latest english news, important news in english from india's leading newspaper re"
8408,madame.lefigaro.fr,Rejet Absolu : Astrologie / Horoscope ('horoscope'),"mode, beauté, recettes, société, horoscope, célébrités"
1909,newsindiatimes.com,Rejet Absolu : Astrologie / Horoscope ('horoscope'),"home - news india times videos itv gold, in association with ani, brings you south asia newsline christmas kar"
1741,telugupeople.com,Rejet Absolu : Astrologie / Horoscope ('horoscope'),"telugupeople.com - andhra pradesh, telangana, news, news analysis, politics, movies, cinema, entertainment, hy"


In [9]:
import pandas as pd
from IPython.display import display

print("⏳ Chargement des données de diagnostic et des features Parquet...")

# 1. Récupération des sources exclues (depuis la mémoire ou le fichier CSV sauvegardé à l'étape précédente)
if "df_exclus" in globals() and len(df_exclus) > 0:
  exclus_ids = df_exclus
else:
  df_audit = pd.read_csv("audit_classification_domaines.csv")
  exclus_ids = df_audit[df_audit["status"] == "❌ Exclus"].copy()

print(f"✔ {len(exclus_ids):,} sources exclues à analyser.")

# 2. Chargement et fusion des fichiers Parquet originaux
df_wiki = pd.read_parquet("features_sources_wiki.parquet").assign(
    Origine="Wiki"
)
df_not_wiki = pd.read_parquet("features_sources_not_in_wiki.parquet").assign(
    Origine="Not-Wiki"
)
df_features_all = pd.concat([df_wiki, df_not_wiki], ignore_index=True)

# 3. Jointure pour récupérer les métriques mathématiques UNIQUEMENT pour les exclus
df_exclus_features = exclus_ids[
    ["SourceCommonName_ID", "domain", "motif"]
].merge(df_features_all, on="SourceCommonName_ID", how="inner")

# On trie par volume d'articles décroissant pour voir les plus "gros" intrus en premier
cols_affichage = [
    "domain",
    "Origine",
    "motif",
    "total_articles",
    "years_active",
    "inactive_days_rate",
    "cv_daily",
    "cv_monthly",
    "mean_wordcount",
    "h_norm_themes",
    "ratio_eco",
    "h_norm_eco",
]
cols_visibles = [
    c for c in cols_affichage if c in df_exclus_features.columns
]
df_exclus_sorted = df_exclus_features[cols_visibles].sort_values(
    by="total_articles", ascending=False
)

print(
    f"\n🔍 TOP 50 DES INTRUS ÉVINCÉS PAR LE SCRAPING (triés par volume"
    f" d'articles) :\n"
    + "=" * 95
)

# 4. Formatage et coloration pour repérer immédiatement quel critère a été "trop permissif"
format_dict = {
    "total_articles": "{:,.0f}",
    "years_active": "{:.1f}",
    "inactive_days_rate": "{:.1%}",
    "cv_daily": "{:.2f}",
    "cv_monthly": "{:.2f}",
    "mean_wordcount": "{:,.0f}",
    "h_norm_themes": "{:.2f}",
    "ratio_eco": "{:.1%}",
    "h_norm_eco": "{:.2f}",
}

styler = df_exclus_sorted.head(50).style.format(format_dict, na_rep="-")

# On colore en VERT les variables qui les ont fait passer (ex: forte diversité ou bon volume)
styler = styler.background_gradient(
    subset=[
        c
        for c in ["h_norm_themes", "total_articles", "years_active"]
        if c in cols_visibles
    ],
    cmap="Blues",
)

# On colore en ROUGE/ORANGE les variables qui auraient dû vous alerter (ex: très faible ratio_eco)
if "ratio_eco" in cols_visibles:
  styler = styler.background_gradient(
      subset=["ratio_eco"], cmap="OrRd", vmin=0, vmax=0.10
  )  # Plus c'est rouge, plus le ratio éco est proche de 0
if "inactive_days_rate" in cols_visibles:
  styler = styler.background_gradient(
      subset=["inactive_days_rate"], cmap="OrRd", vmin=0, vmax=0.30
  )

styler = styler.set_properties(
    subset=["domain", "motif"], **{"font-weight": "bold", "text-align": "left"}
)
display(styler)

# ==============================================================================
# 5. DIAGNOSTIC MACRO : OÙ ÉTAIT LA FAILLE DE SÉPARATION ?
# ==============================================================================
print("\n📊 COMPARAISON DES MÉDIANES : PRESSE LÉGITIME vs EXCLUS DE SCRAPING")
print("=" * 80)

# On sépare les sources retenues en 2 groupes : celles validées par le scraping vs celles exclues
ids_exclus_set = set(exclus_ids["SourceCommonName_ID"])
df_legitimes = df_features_all[
    ~df_features_all["SourceCommonName_ID"].isin(ids_exclus_set)
    & df_features_all["SourceCommonName_ID"].isin(
        set(df_to_scrape["SourceCommonName_ID"])
    )
]

cols_num = [
    "total_articles",
    "years_active",
    "inactive_days_rate",
    "cv_daily",
    "cv_monthly",
    "mean_wordcount",
    "h_norm_themes",
    "ratio_eco",
    "h_norm_eco",
]
cols_num_existantes = [c for c in cols_num if c in df_features_all.columns]

df_comp = pd.DataFrame({
    "✔ Presse Légitime (Médiane)": df_legitimes[cols_num_existantes].median(),
    "❌ Intrus Exclus (Médiane)": df_exclus_features[
        cols_num_existantes
    ].median(),
})
df_comp["Écart / Ratio"] = (
    df_comp["❌ Intrus Exclus (Médiane)"]
    / df_comp["✔ Presse Légitime (Médiane)"]
)

# Affichage propre de la comparaison
format_comp = {
    "✔ Presse Légitime (Médiane)": "{:,.2f}",
    "❌ Intrus Exclus (Médiane)": "{:,.2f}",
    "Écart / Ratio": "{:.1%}",
}
display(
    df_comp.style.format(format_comp).background_gradient(
        subset=["Écart / Ratio"], cmap="coolwarm", vmin=0.5, vmax=1.5
    )
)

⏳ Chargement des données de diagnostic et des features Parquet...
✔ 489 sources exclues à analyser.

🔍 TOP 50 DES INTRUS ÉVINCÉS PAR LE SCRAPING (triés par volume d'articles) :


,domain,Origine,motif,total_articles,years_active,inactive_days_rate,cv_daily,cv_monthly,mean_wordcount,h_norm_themes,ratio_eco,h_norm_eco
32,indiatimes.com,Wiki,Lifestyle / Mode / People (mot: 'fashion'),"4,957,871",12.0,0.4%,0.25,0.19,362,0.57,6.0%,0.46
10,dailymail.co.uk,Wiki,Lifestyle / Mode / People (mot: 'celebrity'),"2,582,749",12.0,0.6%,0.40,0.31,695,0.59,2.4%,0.46
19,philstar.com,Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"1,030,106",11.0,26.6%,1.54,1.42,483,0.58,3.8%,0.49
44,chinadaily.com.cn,Not-Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"987,105",12.0,0.4%,0.47,0.26,319,0.56,6.8%,0.45
103,mirror.co.uk,Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"961,577",12.0,0.4%,0.27,0.19,504,0.57,2.1%,0.38
24,thenews.com.pk,Not-Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"833,214",12.0,0.4%,0.48,0.43,348,0.57,4.1%,0.49
20,hindustantimes.com,Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"800,342",12.0,1.8%,0.69,0.57,415,0.57,2.4%,0.47
18,forbes.com,Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"768,676",12.0,1.2%,0.53,0.31,780,0.58,8.9%,0.42
22,businessinsider.com,Wiki,Lifestyle / Mode / People (mot: 'lifestyle'),"728,346",12.0,0.4%,0.63,0.45,512,0.58,7.8%,0.46
208,prokerala.com,Not-Wiki,Lifestyle / Mode / People (mot: 'astrology'),"720,281",12.0,9.3%,0.48,0.41,314,0.56,3.1%,0.47



📊 COMPARAISON DES MÉDIANES : PRESSE LÉGITIME vs EXCLUS DE SCRAPING


,✔ Presse Légitime (Médiane),❌ Intrus Exclus (Médiane),Écart / Ratio
total_articles,"32,709.00","44,057.00",134.7%
years_active,11.00,10.00,90.9%
inactive_days_rate,0.24,0.30,123.7%
cv_daily,1.01,1.00,98.5%
cv_monthly,0.73,0.81,111.1%
mean_wordcount,383.07,415.23,108.4%
h_norm_themes,0.55,0.56,101.7%
ratio_eco,0.03,0.03,100.9%
h_norm_eco,0.41,0.41,100.4%
